Imports

In [1]:
from typing import Sequence
import multiprocess as mp

import time
import sqlite3
import queue
import threading

import chess.pgn
from stockfish import Stockfish

Set up sqlite db and create if it does not exist

In [4]:
conn = sqlite3.connect('chess.db')

cur = conn.cursor()

cur.execute('''
CREATE TABLE IF NOT EXISTS games (
    game_id INTEGER PRIMARY KEY,
    game_order INTEGER,
    event TEXT,
    site TEXT,
    date_played TEXT,
    round TEXT,
    white TEXT,
    black TEXT,
    result TEXT,
    white_elo INTEGER,
    white_rating_diff INTEGER,
    black_elo INTEGER,
    black_rating_diff INTEGER,
    white_title TEXT,
    black_title TEXT,
    winner TEXT,
    winner_elo INTEGER,
    loser TEXT,
    loser_elo INTEGER,
    winner_loser_elo_diff INTEGER,
    eco TEXT,
    termination TEXT,
    time_control TEXT,
    utc_date TEXT,
    utc_time TEXT,
    variant TEXT,
    ply_count INTEGER,
    date_created TEXT,
    file_name TEXT
);
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS actual_moves (
    game_id       INTEGER,     -- ID mapping to the games file
    move_no       INTEGER,  -- Sequential order of moves
    move_no_pair  INTEGER,  -- Chess move number (e.g., 1, 1, 2, 2...)
    player        TEXT,     -- Player name
    notation      TEXT,     -- Standard Algebraic Notation (e.g., e4, Nf3)
    move          TEXT,     -- UCI format (e.g., e2e4)
    from_square   TEXT,     -- Starting square (e.g., e2)
    to_square     TEXT,     -- Destination square (e.g., e4)
    piece         TEXT,     -- Piece initial (P, N, B, R, Q, K)
    promotion     TEXT,     -- piece if it was promoted
    color         TEXT,     -- "White" or "Black"
    fen_before    TEXT,      -- FEN string of the starting position
    fen_after     TEXT,      -- FEN string of the resulting position
    time_remaining INTEGER, -- Time left on the clock in seconds
    time_spent     REAL,     -- Time spent in seconds 
    static_eval_before         REAL,     -- +/- engine evaluation with no search of the position before the move
    static_eval_after          REAL,     -- +/- engine evaluation with no search of the position after the move
    eval_before           REAL,     -- +/- engine evaluation of the position befpre the move
    mate_count_before     INTEGER,  -- null if the position is not mate + count for white negative for black
    eval_after            REAL,     -- +/- engine evaluation of the position after the move
    mate_count_after      INTEGER,  -- null if the position is not mate + count for white negative for black
    game_to_position      TEXT,   -- Full list of the moves in the game up until this position
    white_win_perc_before REAL,    -- engine evaluation of the win probability before the move
    black_win_perc_before REAL,    -- engine evaluation of the win probability before the move
    draw_perc_before      REAL,     -- engine evaluation of the draw probability before the move
    white_win_perc_after  REAL,    -- engine evaluation of the win probability after the move
    black_win_perc_after  REAL,    -- engine evaluation of the win probability after the move
    draw_perc_after       REAL     -- engine evaluation of the draw probability after the move
);
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS possible_move_evals (
    game_id       INTEGER,     -- ID mapping to the games file
    move_no       INTEGER,  -- Sequential order of moves
    move_no_pair  INTEGER,  -- Chess move number (e.g., 1, 1, 2, 2...)
    notation      TEXT,     -- Standard Algebraic Notation (e.g., e4, Nf3)
    move          TEXT,     -- UCI format (e.g., e2e4)
    from_square   TEXT,     -- Starting square (e.g., e2)
    to_square     TEXT,     -- Destination square (e.g., e4)
    piece         TEXT,     -- Piece initial (P, N, B, R, Q, K)
    promotion     TEXT,     -- piece if it was promoted
    color         TEXT,     -- "White" or "Black"
    fen_before    TEXT,      -- FEN string of the starting position
    fen_after    TEXT,      -- FEN string of the resulting position
    eval          REAL,     -- +/- engine evaluation of the position after the move
    mate_count    INTEGER,  -- null if the position is not mate + count for white negative for black
    white_win_perc REAL,    -- engine evaluation of the win probability after the move
    black_win_perc REAL,    -- engine evaluation of the win probability after the move
    draw_perc      REAL     -- engine evaluation of the draw probability after the move
);
''')

conn.commit()

cur.execute('''CREATE INDEX IF NOT EXISTS index_games_white ON games (white);''')
conn.commit()

cur.execute('''CREATE INDEX IF NOT EXISTS index_games_black ON games (black);''')
conn.commit()

cur.execute('''CREATE INDEX IF NOT EXISTS index_actual_moves_game_id ON actual_moves (game_id);''')
conn.commit()

cur.execute('''CREATE INDEX IF NOT EXISTS index_actual_moves_player ON actual_moves (player);''')
conn.commit()

cur.execute('''CREATE INDEX IF NOT EXISTS index_possiblie_move_evals_game_id ON possible_move_evals (game_id);''')
conn.commit()

conn.close()

Verify stockfish is installed and loading

In [3]:
stockfish = Stockfish(path='C:\\Users\\micha\\Personal\\Coding\\chess-clone\\stockfish\\stockfish-windows-x86-64.exe')
stockfish.set_fen_position("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1") # The starting position
evaluation = stockfish.get_evaluation()
print(f"Position evaluation: {evaluation}")
moves = stockfish.get_top_moves(218) # maximum number of moves ever possible
print(f"Moves evaluation: {moves}")
wdl = stockfish.get_wdl_stats()
print(f"WDL evaluation: {wdl}")
stockfish.send_quit_command()


Position evaluation: {'type': 'cp', 'value': 39}
Moves evaluation: [{'Move': 'e2e4', 'Centipawn': 28, 'Mate': None}, {'Move': 'c2c4', 'Centipawn': 22, 'Mate': None}, {'Move': 'g1f3', 'Centipawn': 22, 'Mate': None}, {'Move': 'g2g3', 'Centipawn': 21, 'Mate': None}, {'Move': 'd2d4', 'Centipawn': 20, 'Mate': None}, {'Move': 'e2e3', 'Centipawn': 16, 'Mate': None}, {'Move': 'd2d3', 'Centipawn': 8, 'Mate': None}, {'Move': 'a2a3', 'Centipawn': 4, 'Mate': None}, {'Move': 'b1c3', 'Centipawn': -1, 'Mate': None}, {'Move': 'c2c3', 'Centipawn': -5, 'Mate': None}, {'Move': 'f2f4', 'Centipawn': -12, 'Mate': None}, {'Move': 'h2h3', 'Centipawn': -18, 'Mate': None}, {'Move': 'a2a4', 'Centipawn': -25, 'Mate': None}, {'Move': 'b2b3', 'Centipawn': -26, 'Mate': None}, {'Move': 'b2b4', 'Centipawn': -30, 'Mate': None}, {'Move': 'h2h4', 'Centipawn': -36, 'Mate': None}, {'Move': 'g1h3', 'Centipawn': -58, 'Mate': None}, {'Move': 'b1a3', 'Centipawn': -70, 'Mate': None}, {'Move': 'f2f3', 'Centipawn': -86, 'Mate': N

Define a class for managing threaded insert requests

In [2]:
class ParallelQueryWriter:
    def __init__(self, db_path="chess.db"):
        self.db_path = db_path
        # mp.Queue and mp.Event are Jupyter-friendly versions
        self.queue = mp.Queue()
        self.stop_event = mp.Event()
        
        # We define the process here
        self.worker = mp.Process(
            target=self._process_queue, 
            args=(self.queue, self.stop_event, self.db_path),
            daemon=True
        )
        self.worker.start()

    def add_query(self, query: str, data: Sequence) -> None:
        """Pushes a query and its data into the parallel queue."""
        self.queue.put((query, data))

    def get_queue_size(self):
        return self.queue.qsize()

    @staticmethod
    def _process_queue(queue, stop_event, db_path):
        # Imports inside the static method for visibility
        import sqlite3
        import multiprocess as mp

        """
        This runs in a completely separate CPU process.
        Static method ensures we don't accidentally pickle the whole class instance.
        """
        # Connection must be opened inside the new process
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        while not stop_event.is_set():
            try:
                # Wait for data (1 second timeout to check stop_event)
                item = queue.get(timeout=1.0)
                query, data = item
                
                cursor.execute("BEGIN")
                cursor.executemany(query, data)
                conn.commit()
            except (mp.queues.Empty, EOFError):
                continue
            except Exception as e:
                # Note: This print usually appears in the terminal/console, 
                # not the Jupyter cell output.
                print(f"Database Process Error: {e}")
                
        conn.close()

    def stop(self):
        """Signals the process to finish and joins the process."""
        if self.worker.is_alive():
            self.stop_event.set()
            self.worker.join()
            print("Database worker stopped.")

In [4]:
class ChessProcessor:
    def __init__(self, query_writer, num_workers=10, engine_path=r'C:\Users\micha\Personal\Coding\chess-clone\stockfish\stockfish-windows-x86-64.exe', depth=15, starting_id=1000000000):
        self.query_writer = query_writer
        self.game_queue = mp.Queue(maxsize=num_workers) # Limit queue size to 10
        self.stop_event = mp.Event()
        self.workers = []

        # Spawn 10 independent Analysis Processes
        for i in range(num_workers):
            # Each worker gets a unique starting ID range or offset if needed, 
            # though usually current_id is managed by the loop logic.
            p = mp.Process(
                target=self._analysis_worker,
                args=(self.game_queue, self.query_writer, self.stop_event, 
                      engine_path, depth, starting_id + (i * 100000)), 
                daemon=True
            )
            p.start()
            self.workers.append(p)

    def add_game(self, game, count):
        """
        This will BLOCK if the queue already has 10 games (one for each worker).
        It perfectly satisfies your 'wait until one frees up' requirement.
        """
        self.game_queue.put((game, count))

    def get_queue_size(self):
        return self.game_queue.qsize()

    @staticmethod
    def _analysis_worker(game_queue, db_queue, stop_event, engine_path, depth, worker_start_id):
        # Imports inside the static method for visibility
        from stockfish import Stockfish
        import multiprocess as mp
        from chess_evaluator import evaluate_game
        
        engine = Stockfish(path=engine_path, depth=depth, parameters= {"Threads": 1, "Minimum Thinking Time": 10, "Ponder": False})
        current_id = worker_start_id

        while not stop_event.is_set():
            try:
                (game, count) = game_queue.get(timeout=1.0)
                # Run evaluation logic (ensure evaluate_game is defined in your notebook)
                evaluate_game(db_queue, engine, game, count) #using count for now as a text
                current_id += 1
            except (mp.queues.Empty, EOFError):
                continue
            except Exception as e:
                print(f"Worker Error: {e}")
        
        engine.send_quit_command()

    def shutdown(self):
        self.stop_event.set()
        for p in self.workers:
            p.join()
        print("All 14 analysis workers stopped.")

Processing games with new parallelization

Run a test on a small pgn

In [ ]:
pgns = open("lichess_db_standard_rated_2025-01.pgn")

start_count = 0
max_count = 15000
cur_game = chess.pgn.read_game(pgns)
count = 9980
depth = 12 #Depth 12 should still be super human and good enough for our purposes
writer = ParallelQueryWriter(db_path="chess.db")
processor = ChessProcessor(writer, num_workers=14, depth=depth, starting_id=0)

while count < start_count:
    cur_game = chess.pgn.read_game(pgns)
    count= count+1
    
try:
    while cur_game is not None and count < max_count:
        processor.add_game(cur_game, count) 
        print(f"Adding game {count} to the pool")
        cur_game = chess.pgn.read_game(pgns)
        count= count+1
    while writer.get_queue_size() > 0 and  processor.get_queue_size() > 0:
        time.sleep(5)
except Exception as e:
    print(f"Unexpected failure: {e}")
finally:
    print(f"Stopping services")
    processor.shutdown()
    writer.stop()

Adding game 0 to the pool
Adding game 1 to the pool
Adding game 2 to the pool
Adding game 3 to the pool
Adding game 4 to the pool
Adding game 5 to the pool
Adding game 6 to the pool
Adding game 7 to the pool
Adding game 8 to the pool
Adding game 9 to the pool
Adding game 10 to the pool
Adding game 11 to the pool
Adding game 12 to the pool
Adding game 13 to the pool
Adding game 14 to the pool
Adding game 15 to the pool
Adding game 16 to the pool
Adding game 17 to the pool
Adding game 18 to the pool
Adding game 19 to the pool
Adding game 20 to the pool
Adding game 21 to the pool
Adding game 22 to the pool
Adding game 23 to the pool
Adding game 24 to the pool
Adding game 25 to the pool
Adding game 26 to the pool
Adding game 27 to the pool
Adding game 28 to the pool
Adding game 29 to the pool
Adding game 30 to the pool
Adding game 31 to the pool
Adding game 32 to the pool
Adding game 33 to the pool
Adding game 34 to the pool
Adding game 35 to the pool
Adding game 36 to the pool
Adding game

Test save headers

In [9]:
conn = sqlite3.connect('chess.db')
cur = conn.cursor()
cur.execute("Select * FROM games where game_id > 9980 limit 10")
conn.commit()
rows = cur.fetchall()
count = 0
for row in rows:
        print(row)
cur.execute("Select * FROM actual_moves where move_no > 15 and  game_id > 9980 limit 10")
conn.commit()
rows = cur.fetchall()
count = 0
for row in rows:
        print(row)
cur.execute("Select *  FROM possible_move_evals where move_no > 15 and  game_id > 9980 limit 10")
conn.commit()
rows = cur.fetchall()
count = 0
for row in rows:
        print(row)
        
conn.commit()
conn.close()